# DeepScaler — train trên Google Colab / Kaggle

Notebook này chạy phần **huấn luyện** của DeepScaler (AdapGLA + học đồ thị thích nghi) cho benchmark **Online Boutique** (10 service × 4 đặc trưng `pod, cpu, res, req`; mục tiêu = số pod ở bước kế tiếp).

**Nguồn dữ liệu** (chọn ở ô cấu hình):
- `"cluster"` — CSV thật thu từ Prometheus bằng `pipeline/collect.py` trên cluster Kubernetes + Istio (xem `deploy/README.md`). Upload file CSV lên.
- `"simulated"` — dữ liệu **mô phỏng** (`pipeline/simulate.py`, phát lại trace FIFA'98 qua call graph của Boutique + bộ điều khiển kiểu HPA). Dùng để chạy thử pipeline khi chưa có cluster; **không** dùng làm kết quả thực nghiệm trong luận văn.

Colab: *Runtime → Change runtime type → GPU (T4)*. Kaggle: *Settings → Accelerator → GPU*, bật *Internet*.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL    = "https://github.com/maintn24/DeepScaler.git"   # repo đã push thư mục pipeline/ và train.py
BRANCH      = "deepscaler-pipeline"
DATA_SOURCE = "cluster"        # "cluster" (dữ liệu thật) | "simulated" (chỉ để thử pipeline)
RAW_CSV     = None             # "cluster": đường dẫn CSV, ví dụ "/content/run2_hpa_5f.csv" (None = hỏi upload trên Colab)
OD_ADJ      = None             # "cluster": file *_od.npy từ collect.py (None = call graph tĩnh của Boutique)
SPLIT       = "0.6,0.2,0.2"    # train/valid/test, chia theo thời gian
STEP_IN     = 20               # độ dài lịch sử (số bước). Repo gốc dùng 80
NUM_EPOCH   = 15               # số epoch tối đa mỗi vòng
NUM_ITER    = 6                # số vòng luân phiên (predictor <-> graph learner)
SIM_LOOPS   = 16               # mô phỏng: số lần phát lại trace (75 phút/lần)

In [ ]:
import os, sys, subprocess
IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB  = "google.colab" in sys.modules
WORK = "/kaggle/working" if IN_KAGGLE else "/content"
os.chdir(WORK)
if not os.path.exists("DeepScaler"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "DeepScaler"], check=True)
os.chdir(f"{WORK}/DeepScaler")
assert os.path.exists("train.py") and os.path.exists("pipeline"), \
    "Repo chưa có train.py / pipeline/: hãy commit + push các file mới trước (hoặc upload zip rồi giải nén vào DeepScaler/)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| cwd:", os.getcwd())

## 1. Dữ liệu thô

In [ ]:
if DATA_SOURCE == "simulated":
    RAW_CSV = "data/raw/simulated.csv"
    !python -m pipeline.simulate --loops {SIM_LOOPS} --interval 30 --out {RAW_CSV}
else:
    if RAW_CSV is None and IN_COLAB:
        from google.colab import files
        up = files.upload()                       # chọn file CSV từ collect.py
        RAW_CSV = list(up.keys())[0]
    assert RAW_CSV and os.path.exists(RAW_CSV), "Chỉ định RAW_CSV (Kaggle: /kaggle/input/<dataset>/<file>.csv)"
DATA_DIR = f"data/boutique_{DATA_SOURCE}"
OUT_DIR  = f"model_states/boutique_{DATA_SOURCE}"
import pandas as pd
raw = pd.read_csv(RAW_CSV); print(raw.shape); raw.filter(like="frontend").describe().round(3)

## 2. Tạo tập train / valid / test (chia theo thời gian theo SPLIT)

In [ ]:
!python -m pipeline.build_dataset --raw {RAW_CSV} --out_dir {DATA_DIR} --step_in {STEP_IN} --split {SPLIT} {"--od_adj " + OD_ADJ if OD_ADJ else ""}

## 3. Chạy thử nhanh (1 epoch, 1 vòng) để chắc pipeline không lỗi

In [ ]:
!python train.py --data_dir {DATA_DIR} --out_dir model_states/smoke --num_epoch 1 --num_iter 1

## 4. Huấn luyện đầy đủ

In [ ]:
!python train.py --data_dir {DATA_DIR} --out_dir {OUT_DIR} --num_epoch {NUM_EPOCH} --num_iter {NUM_ITER}

## 5. Kết quả trên tập test

In [ ]:
import json, numpy as np
m = json.load(open(f"{OUT_DIR}/metrics.json"))
rows = {k: v for k, v in m["results"]["test"].items()}
df = pd.DataFrame(rows).T[["mae","rmse","mape","pods_exact_match","under_provision_rate","over_provision_rate","avg_pods_pred","avg_pods_true"]]
print("Thời gian train: %.0f s trên %s" % (m["train_time_s"], m["device"]))
print("Đồ thị học được:", m["graph"])
df.round(3)

In [ ]:
import matplotlib.pyplot as plt
P = np.load(f"{OUT_DIR}/predictions.npz")
x_test = np.load(f"{DATA_DIR}/test.npz")["x"]
services = json.load(open(f"{DATA_DIR}/meta.json"))["services"]
show = ["frontend", "recommendationservice", "cartservice", "productcatalogservice"]
fig, axes = plt.subplots(len(show), 1, figsize=(11, 2.4*len(show)), sharex=True)
for ax, s in zip(axes, show):
    j = services.index(s)
    ax.step(range(len(P["test_true"])), P["test_true"][:, 0, j], where="post", color="#2a78d6", lw=2, label="Pod thực tế")
    ax.plot(P["test"][:, 0, j], color="#eb6834", lw=1.5, label="DeepScaler dự đoán")
    ax.set_title(s, loc="left", fontsize=10); ax.grid(alpha=.25); ax.spines[["top","right"]].set_visible(False)
axes[0].legend(loc="upper right", frameon=False, ncol=2)
axes[-1].set_xlabel("Bước thời gian (tập test)")
plt.tight_layout(); plt.show()

In [ ]:
adj = np.load(f"{OUT_DIR}/AdapGLA/best_adj_mx.npy"); od = np.load(f"{DATA_DIR}/adj_od.npy")
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for a, M, t in [(ax[0], od + od.T, "Call graph (Istio / Boutique)"), (ax[1], adj, "Đồ thị DeepScaler học được")]:
    im = a.imshow(M, cmap="Blues"); a.set_title(t, fontsize=10)
    a.set_xticks(range(len(services))); a.set_xticklabels([s.replace("service","") for s in services], rotation=60, fontsize=8)
    a.set_yticks(range(len(services))); a.set_yticklabels([s.replace("service","") for s in services], fontsize=8)
    fig.colorbar(im, ax=a, fraction=.046)
plt.tight_layout(); plt.show()

## 6. Tải model về (dùng cho `predict_scale.py` / bộ autoscaler trên cluster)

In [ ]:
import shutil
zip_path = shutil.make_archive(f"{WORK}/deepscaler_{DATA_SOURCE}", "zip", ".", OUT_DIR)
shutil.copy(f"{DATA_DIR}/meta.json", f"{WORK}/meta_{DATA_SOURCE}.json")
print(zip_path)
if IN_COLAB:
    from google.colab import files; files.download(zip_path)
# Kaggle: file nằm trong /kaggle/working -> tab Output